# Web Scraping con Selenium



Importación de las librerías

In [2]:
# Para la manipulación de datos
import pandas as pd
import numpy as np

# Servicio y driver de Chrome de Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Las opciones que vamos a tener para buscar elementos
from selenium.webdriver.common.by import By

# Para cuando queramos mandar pulsaciones de teclado
from selenium.webdriver.common.keys import Keys

# Hacemos que espere
import time

# Importaciones para esperas explícitas (mejor práctica que time.sleep)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Importamos undetected-chromedriver para evitar el captcha
import undetected_chromedriver as uc

# Importamos excepciones comunes de Selenium para manejo de errores
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [3]:
service = Service(ChromeDriverManager().install())

In [4]:
from selenium import webdriver

driver = webdriver.Chrome()
print(driver.capabilities["browserVersion"])
driver.quit()

146.0.7680.178


Creamos el driver para controlar el navegador

In [5]:
# Aquí empiezo ejecutando el robot

import undetected_chromedriver as uc

driver = uc.Chrome(
    version_main=146,  #Para que funcione con chrome 146
    browser_executable_path=r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    service=service,
    use_subprocess=False,
    headless=False,
)

Accedemos a la página principal

In [6]:
url = "https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/lo-barnechea/0"
driver.get(url)

Hay un botón de Consentir datos personales

In [7]:
wait = WebDriverWait(driver, 10)

accept = wait.until(EC.element_to_be_clickable((
    By.XPATH, "//p[contains(@class, 'fc-button-label') and contains(text(), 'Consentir')]"
)))
accept.click()

Hago una funcion para guardar los datos más importantes de cada propiedad: título, 

In [ ]:
URL_BASE = "https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/lo-barnechea/{}"
TOTAL_PAGINAS = 5  # número total de páginas, HAY QUE MODIFICARLO PARA CADA AYUNTAMIENTO

todos_los_datos = []

for pagina in range(0, TOTAL_PAGINAS):
    driver.get(URL_BASE.format(pagina))
    
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.clp-publication-element-description-container")))

    publicaciones = driver.find_elements(By.CSS_SELECTOR, "div.clp-publication-element-description-container")

    for pub in publicaciones:
        try:
            titulo = pub.find_element(By.CSS_SELECTOR, "h2.publication-title-list a").text.strip()
        except:
            titulo = "N/A"
        try:
            elemento_precio = pub.find_element(By.CSS_SELECTOR, "a.clp-big-value")
            precio = driver.execute_script("return arguments[0].innerText;", elemento_precio).strip()
        except:
            precio = "N/A"
        try:
            dormitorios = pub.find_element(By.CSS_SELECTOR, "span[title='Habitaciones'] span.clp-feature-value").text.strip()
        except:
            dormitorios = "N/A"
        try:
            banos = pub.find_element(By.CSS_SELECTOR, "span[title='Baños'] span.clp-feature-value").text.strip()
        except:
            banos = "N/A"
        try:
            m2 = pub.find_element(By.CSS_SELECTOR, "span.clp-feature-value[title='Superficie Total']").text.strip()
        except:
            m2 = "N/A"
        try:
            codigo_elemento = pub.find_element(By.XPATH, ".//span[@class='light-bold' and contains(text(),'Código')]/..")
            codigo = codigo_elemento.text.replace("Código:", "").strip()
        except:
            codigo = "N/A"
        try:
            estacionamientos = pub.find_element(By.XPATH, ".//span[@class='clp-feature-description' and contains(text(),'Estacionamientos')]/following-sibling::span[@class='clp-feature-value']").text.strip()
        except:
            estacionamientos = "N/A"
        try:
            padre = driver.execute_script("return arguments[0].parentElement.parentElement;", pub)
            fecha = padre.find_element(By.CSS_SELECTOR, "div.clp-publication-date small").text.strip()
        except:
            fecha = "N/A"

        todos_los_datos.append({
            "titulo": titulo,
            "precio": precio,
            "dormitorios": dormitorios,
            "baños": banos,
            "m2": m2,
            "codigo": codigo,
            "estacionamientos": estacionamientos,
            "fecha": fecha
        })

    print(f"Página {pagina + 1}/{TOTAL_PAGINAS} — total registros: {len(todos_los_datos)}")
    time.sleep(1)

print(f"✅ Total propiedades extraídas: {len(todos_los_datos)}")

Página 1/5 — total registros: 10
Página 2/5 — total registros: 20
Página 3/5 — total registros: 30
Página 4/5 — total registros: 40
Página 5/5 — total registros: 47
✅ Total propiedades extraídas: 47


In [9]:
df_Pisos = pd.DataFrame(todos_los_datos)
df_Pisos.head()

,titulo,precio,dormitorios,baños,m2,codigo,estacionamientos,fecha
0,"Lo Barnechea, Valle del Monasterio",$ 2.390.503,5,5,220 m²,26612163\nArriendo Mensual / Departamento / Lo...,2,07/03/2026
1,"Lo Barnechea, El rodeo Publicación Reciente",$ 1.673.352,3,2,150 m²,114822532\nArriendo Mensual / Departamento / L...,3,07/04/2026
2,"Lo Barnechea, Avenida San Josemaría Escrivá de...",$ 560.000,2,1,55 m²,10670006\nArriendo Mensual / Departamento / Lo...,1,05/04/2026
3,"Lo Barnechea, Moderno en La Dehesa (156198)",$ 956.201,1,1,52 m²,114523717\nArriendo Mensual / Departamento / L...,1,04/04/2026
4,"Lo Barnechea, David Ben Gurion",$ 1.195.252,2,2,72 m²,114505801\nArriendo Mensual / Departamento / L...,1,04/04/2026


In [10]:
import numpy as np

# Filtro el df y elimino la información que no sirve

df_Pisos['titulo'] = df_Pisos['titulo'].str.replace('Las Condes, ', '', regex=False).str.replace('Publicación Reciente', '', regex=False).str.strip()

# Sólo guardo el código del departamento y elimino el resto
df_Pisos['codigo'] = df_Pisos['codigo'].str.split('\n').str[0].str.strip()

# Cambio los N/A de estacionamientos y los relleno con cero 
df_Pisos["estacionamientos"] = df_Pisos["estacionamientos"].replace("N/A", np.nan)
df_Pisos["estacionamientos"] = df_Pisos["estacionamientos"].fillna(0)
df_Pisos.head(5)

,titulo,precio,dormitorios,baños,m2,codigo,estacionamientos,fecha
0,"Lo Barnechea, Valle del Monasterio",$ 2.390.503,5,5,220 m²,26612163,2,07/03/2026
1,"Lo Barnechea, El rodeo",$ 1.673.352,3,2,150 m²,114822532,3,07/04/2026
2,"Lo Barnechea, Avenida San Josemaría Escrivá de...",$ 560.000,2,1,55 m²,10670006,1,05/04/2026
3,"Lo Barnechea, Moderno en La Dehesa (156198)",$ 956.201,1,1,52 m²,114523717,1,04/04/2026
4,"Lo Barnechea, David Ben Gurion",$ 1.195.252,2,2,72 m²,114505801,1,04/04/2026


In [11]:
df_Pisos['codigo'] = df_Pisos['codigo'].str.split('\n').str[0].str.strip()

In [12]:
# CAMBIAR EL NOMBRE DEL ARCHIVO!!

In [ ]:
df_Pisos.to_csv(r"C:\Users\Ariela\Desktop\EDA - Inmobiliario\Inmobiliario\EDA_Inmob_Lobarnechea.csv", index=False, encoding="utf-8-sig")